# RQ3.2: Which parameters can help estimate energy consumption?

Author: Santiago del Rey


## Import libraries


In [9]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.metrics import root_mean_squared_error
from tqdm import tqdm

from src.environment import CONFIGS_DIR, DATA_DIR, FIGURES_DIR, METRICS_DIR
from src.features.analysis import build_energy_estimation, find_stabilizing_point, plot_regime_change
# from src.features.preprocessing import (
#     HOURS_TO_SECONDS,
#     JOULES_TO_KJOULES,
#     KJOULES_TO_JOULES,
#     MJOULES_TO_KJOULES,
#     get_epoch_ends,
# )

# SAVE_FIGS = True
# FIGURES_FORMAT = "pdf"
# SAVE_FIGS_DIR = FIGURES_DIR / "RQ3"

# if not SAVE_FIGS_DIR.exists():
#     os.makedirs(SAVE_FIGS_DIR)

# # Set numpy random seed for reproducibility
# rng = np.random.default_rng(seed=2024)

# sns.set_theme(style="whitegrid", context="paper", palette="colorblind", color_codes=True, font_scale=1.5)
# plt.style.use(CONFIGS_DIR / "figures.mplstyle")

# %matplotlib inline

: 

## Utility functions and constants


In [ ]:
INDEPENDENT_VARIABLES = [
    "training environment",
    "architecture",
    "dataset",
    "batch size",
    "image size",
    "total ram (GB)",
]
RELEVANT_METRICS = [
    "run_id",
    "start time",
    "training duration (h)",
    "return code",
    "gpu usage (%)",
    "average gpu power (W)",
    "gpu energy (MJ)",
    "max power limit (W)",
    "average ram power (W)",
    "ram energy (MJ)",
    "energy (MJ)",
    "GFLOPs",
    "trained epochs",
    "measured epochs",
    "total seen images",
]

## Load the datasets


In [ ]:
aggregated_metrics = pd.read_parquet(
    METRICS_DIR / "processed" / "clean-dl-training-energy-consumption-dataset.gzip",
    columns=INDEPENDENT_VARIABLES + RELEVANT_METRICS,
).sort_values(by=["start time"])

aggregated_metrics.replace(
    {
        "Desktop Normal User": "Desktop N",
        "Desktop ML Engineer": "Desktop ML",
    },
    inplace=True,
)

aggregated_metrics.head()

In [ ]:
metrics = pd.read_parquet(os.path.join(METRICS_DIR, "interim", "dl-training-profiling-dataset.gzip"))
metrics.query("`run_id` in @aggregated_metrics['run_id'].values", inplace=True)
metrics["elapsed_time"] = metrics["elapsed_time"] / np.timedelta64(1, "s")
metrics["epoch"] = metrics["epoch"].astype("int")

metrics.head()

## Is epoch duration stable?

First, we compute the duration of each epoch using the end time of the epoch and the start time of the next epoch. Then, we plot the distribution of the duration of the epochs.


In [ ]:
epoch_energy_df = pd.read_parquet(METRICS_DIR / "processed" / "clean-dl-epoch-energy-consumption-dataset.gzip")
epoch_energy_df["epoch"] = epoch_energy_df["epoch"].astype("int")
epoch_energy_df.replace(
    {
        "Desktop Normal User": "Desktop N",
        "Desktop ML Engineer": "Desktop ML",
    },
    inplace=True,
)

In [ ]:
epoch_energy_df.head()

From the histogram, we can have an intuition that the epoch duration is generally stable, whithin a small margin.


### Histogram of epoch duration for the Caltech101 dataset in the Server


In [ ]:
hist = epoch_energy_df.query("`training environment` == 'Server' and `dataset` == 'caltech101'").hist(
    column="duration (s)", by=["dataset", "batch size", "image size"], bins=20, figsize=(15, 10), layout=(3, 3)
)
for ax in hist.flatten():
    old_title = ax.get_title()
    if old_title == "":
        continue
    # ax.set_xlabel("Log$_{10}$(Average epoch time) (s)")
    ax.set_xlabel("Average epoch time (s)")
    ax.set_ylabel("Frequency")
    _, batch, input_size0, input_size1 = old_title.split(", ")
    ax.set_title(f"Batch size: {batch}, Image size: {input_size0}, {input_size1[:-1]}")

#### Histogram of epoch duration for the Caltech101 dataset in the Desktop ML Engineer environment


In [ ]:
hist = epoch_energy_df.query("`training environment` == 'Desktop ML' and `dataset` == 'caltech101'").hist(
    column="duration (s)", by=["dataset", "batch size", "image size"], bins=20, figsize=(15, 10), layout=(3, 3)
)
for ax in hist.flatten():
    old_title = ax.get_title()
    if old_title == "":
        continue
    # ax.set_xlabel("Log$_{10}$(Average epoch time) (s)")
    ax.set_xlabel("Average epoch time (s)")
    ax.set_ylabel("Frequency")
    _, batch, input_size0, input_size1 = old_title.split(", ")
    ax.set_title(f"Batch size: {batch}, Image size: {input_size0}, {input_size1[:-1]}")

### Histogram of epoch duration for the Stanford Dogs dataset in the Server


In [ ]:
hist_data = epoch_energy_df.query("`training environment` == 'Server' and `dataset` == 'stanford_dogs' and epoch >= 10")
hist_data["label"] = "Batch: " + hist_data["batch size"].astype(str) + ", Input: " + hist_data["image size"].astype(str)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.boxplot(data=hist_data, y="label", x="duration (s)", ax=ax, showfliers=False)
ax.set_ylabel("")
ax.set_xlabel("Epoch duration (s)")

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"epoch-time-boxplot-server-stanford-dogs.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

#### Histogram of epoch duration for the Stanford Dogs dataset in the Desktop ML Engineer environment


In [ ]:
hist = epoch_energy_df.query("`training environment` == 'Desktop ML' and `dataset` == 'stanford_dogs'").hist(
    column="duration (s)", by=["dataset", "batch size", "image size"], bins=20, figsize=(15, 10), layout=(3, 3)
)
for ax in hist.flatten():
    old_title = ax.get_title()
    if old_title == "":
        continue
    # ax.set_xlabel("Log$_{10}$(Average epoch time) (s)")
    ax.set_xlabel("Average epoch time (s)")
    ax.set_ylabel("Frequency")
    _, batch, input_size0, input_size1 = old_title.split(", ")
    ax.set_title(f"Batch size: {batch}, Image size: {input_size0}, {input_size1[:-1]}")

In [ ]:
from numpy import quantile

epoch_energy_df_no_outliers = epoch_energy_df.groupby(
    ["training environment", "architecture", "dataset", "batch size", "image size"]
).apply(
    lambda x: x.assign(
        q1=quantile(x["duration (s)"], 0.25),
        q3=quantile(x["duration (s)"], 0.75),
        IQR=quantile(x["duration (s)"], 0.75) - quantile(x["duration (s)"], 0.25),
        is_outlier=(
            (
                x["duration (s)"]
                < quantile(x["duration (s)"], 0.25)
                - 1.5 * (quantile(x["duration (s)"], 0.75) - quantile(x["duration (s)"], 0.25))
            )
            | (
                x["duration (s)"]
                > quantile(x["duration (s)"], 0.75)
                + 1.5 * (quantile(x["duration (s)"], 0.75) - quantile(x["duration (s)"], 0.25))
            )
        ),
    )
)

epoch_energy_df_no_outliers = epoch_energy_df_no_outliers.query("is_outlier == False").reset_index(drop=True)

If we look at the at the actual values. We see that most of the maximum differences are below 5 seconds when excluding the first epoch.


In [ ]:
epoch_energy_df.query("`dataset` != 'chesslive-occupancy' and `epoch` > 0").groupby(
    ["training environment", "dataset", "batch size", "image size"]
)["duration (s)"].agg(np.ptp)

In [ ]:
epoch_energy_df.query("`dataset` != 'chesslive-occupancy'").groupby(
    ["training environment", "dataset", "batch size", "image size"]
)["duration (s)"].describe()

## Power profile analysis


### When does power consumption stabilize?


To detect when the power consumption stabilizes, we will apply semanting segmentation on the power profiles of each run.
To this end, we will compute the Matrix Profile (MP) of each run setting the window size $m = 10$.

We will use the [stumpy](https://stumpy.readthedocs.io/en/latest/) library to compute the MP and to apply semantic segmentation.

First, we compute the MP of each run. Then, we use the MP to compute the corrected arc curve (CAC).
From the CAC, we select the point with the lowest value as the point where there is a change in the power consumption regime.
We will call this point the _stabilizing point_.
We will use this point to compute the time it took for the power consumption to stabilize and which epoch it corresponds to.

For this analysis we will use the runs from the Server enviroment using the InceptionV3 model, since are the ones where we have the actual epoch duration.


In [ ]:
m = 10
L = m
if not os.path.exists(DATA_DIR / "analysis" / "processed" / "regimes.gzip") or not os.path.exists(
    DATA_DIR / "analysis" / "processed" / "matrix-profiles.pkl"
):
    regimes_df, profiles = find_stabilizing_point(epoch_energy_df["run_id"].unique(), metrics, m, L, save=True)
    regimes_df["stabilizing epoch"] = regimes_df["stabilizing epoch"].astype(np.uint8)
else:
    regimes_df = pd.read_parquet(DATA_DIR / "analysis" / "processed" / "regimes.gzip")
    regimes_df["stabilizing epoch"] = regimes_df["stabilizing epoch"].astype(np.uint8)
    with open(DATA_DIR / "analysis" / "processed" / "matrix-profiles.pkl", "rb") as f:
        profiles = pickle.load(f)

In [ ]:
regimes_df = regimes_df.merge(
    aggregated_metrics[INDEPENDENT_VARIABLES + ["run_id"]], on="run_id", how="inner"
)  # Add independent variables to group results by batch size and image size
regimes_df.groupby(["dataset", "batch size", "image size"]).describe()

Looking at the histogram of the time it took for the power consumption to stabilize, we can see that over 60% of the runs stabilized in less than a minute from the start of the run.


In [ ]:
axs = regimes_df.hist(column="elapsed time", bins=100, figsize=(15, 6), by="training environment", layout=(3, 1))
for ax in axs.flatten():
    ax.bar_label(ax.containers[0], fmt="%d")
    training_env = ax.get_title()
    stop = regimes_df.query("`training environment` == @training_env")["elapsed time"].max() + 5

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"elapsed-time-histograms.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

Although the elapsed time offers an interesting view in the time it takes for the power consumption to stabilize, it is not reliable to compare runs with different hyperparameters like batch size and image size.

If we look at the epoch where the power consumption stabilized, we can see that more than 90% of the runs stabilized after the fourth epoch.
In fact, if we visually inspect those runs that stabilized after the fourth epoch, we can see there are a couple of false negatives in the the runs that stabilized during epoch 102 and 121.

Based on this results, we will assume that the power consumption stabilizes after the fourth epoch in general.


In [ ]:
axs = regimes_df.hist(column="stabilizing epoch", bins=100, figsize=(15, 10), by="training environment", layout=(3, 1))
for ax in axs.flatten():
    ax.bar_label(ax.containers[0], fmt="%d")
    training_env = ax.get_title()
    stop = regimes_df.query("`training environment` == @training_env")["stabilizing epoch"].max() + 5
    ax.set_xticks(np.arange(0, stop, 5))

axs[-1].set_xlabel("Stabilizing epoch")

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"stabilizing-epoch-histograms.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
_, ax = plt.subplots(figsize=(10, 2.75))

linestyles = {
    "Desktop N": "solid",
    "Desktop ML": "dotted",
    "Server": "dashdot",
}

for training_env in regimes_df["training environment"].unique():
    regimes_df.query("`training environment` == @training_env")["stabilizing epoch"].value_counts(
        normalize=True
    ).sort_index().cumsum().plot(
        label=training_env,
        ax=ax,
        linestyle=linestyles[training_env],
    )
regimes_df["stabilizing epoch"].value_counts(normalize=True).sort_index().cumsum().plot(
    label="All", ax=ax, linestyle="dashed"
)
ax.hlines(0.9, 0, regimes_df["stabilizing epoch"].max() + 5, colors="k", linestyles="dashed", label="90%")
# ax.set_title("Accumulated percentage of runs that stabilize after the n-th epoch")
ax.set_ylabel("Cumulative percentage\nof training executions")
ax.set_xlabel("n-th epoch")
ax.set_xticks(np.arange(0, regimes_df["stabilizing epoch"].max() + 5, 10))
ax.legend()

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"stabilizing-epoch-cumulative-distribution.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
def plot_regime_change_example(stabilizing_epoch):
    example_run = regimes_df.query("`stabilizing epoch` == @stabilizing_epoch")["run_id"].sample(1).iloc[0]
    run = metrics.loc[metrics["run_id"] == example_run]
    epoch_ends = get_epoch_ends(run.iloc[0])
    regime_change = profiles[example_run]["regime_locations"][0]
    cac = profiles[example_run]["cac"]
    plot_regime_change(run, epoch_ends, cac, regime_change, stabilizing_epoch)

In [ ]:
regimes_df

In [ ]:
plot_regime_change_example(0)

In [ ]:
plot_regime_change_example(1)

In [ ]:
plot_regime_change_example(2)

In [ ]:
plot_regime_change_example(3)

In [ ]:
plot_regime_change_example(9)

In [ ]:
plot_regime_change_example(10)

In [ ]:
plot_regime_change_example(15)

In [ ]:
plot_regime_change_example(102)

In [ ]:
plot_regime_change_example(121)

#### Can we accurately estimate the energy consumption of a run based on some initial epochs?

In the previous section, we saw that the power consumption usually stabilizes after the first ten epochs.
Here we will try to estimate the energy consumption of a run based on the power consumption of a few epochs after the stabilizing point.


In [ ]:
stabilizing_epoch = 10

if not os.path.exists(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_window.pkl"):
    groups = metrics.groupby("run_id")
    mean_power_draw = {
        "run_id": [],
        "window size": [],
        "mean gpu power draw": [],
        "mean ram power draw": [],
        "energy (J)": [],
    }
    for run_id, group in tqdm(groups, total=len(groups)):
        n_epochs = group.query("`epoch` >= @stabilizing_epoch")["epoch"].nunique()
        for window_size, step in enumerate(np.arange(n_epochs), start=1):
            data = group.query("`epoch` >= @stabilizing_epoch and `epoch` <= (@stabilizing_epoch + @step)")
            mean_gpu_power = data["gpu_power_draw"].mean()
            mean_ram_power = data["memory_power_draw"].mean()

            epoch_data = epoch_energy_df.query(
                "`run_id` == @run_id and `epoch` >= @stabilizing_epoch and `epoch` <= (@stabilizing_epoch + @step)"
            )
            energy = epoch_data["total energy (kJ)"].sum() * KJOULES_TO_JOULES

            mean_power_draw["run_id"].append(run_id)
            mean_power_draw["window size"].append(window_size)
            mean_power_draw["mean gpu power draw"].append(mean_gpu_power)
            mean_power_draw["mean ram power draw"].append(mean_ram_power)
            mean_power_draw["energy (J)"].append(energy)

    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_window.pkl", "wb") as f:
        pickle.dump(mean_power_draw, f)
else:
    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_window.pkl", "rb") as f:
        mean_power_draw = pickle.load(f)

In [ ]:
energy_estimation_from_first_epochs = build_energy_estimation(mean_power_draw, stabilizing_epoch)
energy_estimation_from_first_epochs.to_parquet(
    DATA_DIR / "analysis" / "processed" / "energy-estimation-from-first-epochs.gzip",
    compression="gzip",
    index=False,
)
energy_estimation_from_first_epochs.head()

In the following figure, we can see how the Root Mean Squared Error (RMSE) decreases as we increase the number of epochs used to estimate the energy consumption. This is to be expected since the more epochs we use, the more accurate the power consumption estimation will be.
However, the relevant point here is that with only 1 epoch after the stabilizing point, we can estimate the energy consumption with an RMSE of less than 0.01.


In [ ]:
_, (ax0, ax1) = plt.subplots(2, 1, figsize=(15, 5))

labels = {"desktop": "Desktop N", "desktop-v2": "Desktop ML", "server": "Server"}
for i, train_environment in enumerate(energy_estimation_from_first_epochs["training environment"].unique()):
    data = energy_estimation_from_first_epochs.query("`training environment` == @train_environment")
    rmse_method_1 = data.groupby("window size")[["energy (kJ)", "estimated energy (kJ) (STEP-P)"]].apply(
        lambda g: root_mean_squared_error(g["energy (kJ)"], g["estimated energy (kJ) (STEP-P)"])
    )
    rmse_method_2 = data.groupby("window size")[["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"])
    )

    ax0.plot(rmse_method_1.index, rmse_method_1, label=labels[train_environment])
    ax1.plot(rmse_method_2.index, rmse_method_2, label=labels[train_environment])

rmse_method_1 = energy_estimation_from_first_epochs.groupby("window size")[
    ["energy (kJ)", "estimated energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["energy (kJ)"], g["estimated energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"]))
ax0.plot(rmse_method_1.index, rmse_method_1, label="All", color="r", linestyle="dashed")
ax1.plot(rmse_method_2.index, rmse_method_2, label="All", color="r", linestyle="dashed")

ax0.legend()
ax0.set_title("RMSE of the energy estimation vs. window size (STEP-P)")
ax0.set_ylabel("Stable epochs error")
ax1.set_ylabel("Full run error")
ax1.set_xlabel("Window size")
stop = energy_estimation_from_first_epochs["window size"].max() + 1
ax0.set_xticks(np.arange(0, stop, 5))
ax1.set_xticks(np.arange(0, stop, 5))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-vs-window-size-method-1.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
_, (ax0, ax1) = plt.subplots(2, 1, figsize=(15, 5))

for i, train_environment in enumerate(energy_estimation_from_first_epochs["training environment"].unique()):
    data = energy_estimation_from_first_epochs.query("`training environment` == @train_environment")
    rmse_method_1 = data.groupby("window size")[["energy (kJ)", "estimated energy (kJ) (STEP-E)"]].apply(
        lambda g: root_mean_squared_error(g["energy (kJ)"], g["estimated energy (kJ) (STEP-E)"])
    )
    rmse_method_2 = data.groupby("window size")[["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"])
    )
    ax0.plot(rmse_method_1.index, rmse_method_1, label=labels[train_environment])
    ax1.plot(rmse_method_2.index, rmse_method_2, label=labels[train_environment])

rmse_method_1 = energy_estimation_from_first_epochs.groupby("window size")[
    ["energy (kJ)", "estimated energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["energy (kJ)"], g["estimated energy (kJ) (STEP-E)"]))
rmse_method_2 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"]))
ax0.plot(rmse_method_1.index, rmse_method_1, label="All", color="r", linestyle="dashed")
ax1.plot(rmse_method_2.index, rmse_method_2, label="All", color="r", linestyle="dashed")

ax0.legend()
ax0.set_title("RMSE of the energy estimation vs. window size (STEP-E)")
ax0.set_ylabel("Stable epochs error")
ax1.set_ylabel("Full run error")
ax1.set_xlabel("Window size")
stop = energy_estimation_from_first_epochs["window size"].max() + 1
ax0.set_xticks(np.arange(0, stop, 5))
ax1.set_xticks(np.arange(0, stop, 5))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-vs-window-size-method-2.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1)

for i, train_environment in enumerate(energy_estimation_from_first_epochs["training environment"].unique()):
    data = energy_estimation_from_first_epochs.query("`training environment` == @train_environment")
    rmse_method_1 = data.groupby("window size")[["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"])
    )
    rmse_method_2 = data.groupby("window size")[["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"])
    )
    ax0.plot(rmse_method_1.index, rmse_method_1, label=labels[train_environment])
    ax1.plot(rmse_method_2.index, rmse_method_2, label=labels[train_environment])

rmse_method_1 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"]))
ax0.plot(rmse_method_1.index, rmse_method_1, label="All", color="r", linestyle="dashed")
ax1.plot(rmse_method_2.index, rmse_method_2, label="All", color="r", linestyle="dashed")

ax0.legend()
ax0.set_title("STEP-P")
ax0.set_ylabel("RMSE")
ax1.set_title("STEP-E")
ax1.set_ylabel("RMSE")
ax1.set_xlabel("window size")
stop = energy_estimation_from_first_epochs["window size"].max() + 1
ax0.set_xticks(np.arange(0, stop, 5))
ax1.set_xticks(np.arange(0, stop, 5))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-vs-window-size.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

rmse_method_1 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"]))
ax.plot(rmse_method_1.index, rmse_method_1, label="power-based")
ax.plot(rmse_method_2.index, rmse_method_2, label="epoch-energy-based")

ax.legend()
ax.set_ylabel("RMSE")
ax.set_xlabel("window size")
stop = energy_estimation_from_first_epochs["window size"].max() + 5
ax.set_xticks(np.arange(0, stop, 5))
ax.set_xticks(np.arange(0, stop, 5))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-vs-window-size-merged.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
window_size = 5
y_data = energy_estimation_from_first_epochs.query("`window size` == 5")
y_true = y_data["total energy (kJ)"]
y_pred = y_data["estimated total energy (kJ) (STEP-P)"]
method_1_rmse_from_first_epochs = root_mean_squared_error(y_true, y_pred)
# (window mean gpu power + mean ram power) * training duration
print(f"RMSE (STEP-P estimator) with a window size of {window_size}: {method_1_rmse_from_first_epochs}")

In [ ]:
y_pred = y_data["estimated total energy (kJ) (STEP-E)"]
# (window total energy * window_size/#epochs)
method_2_rmse_from_first_epochs = root_mean_squared_error(y_true, y_pred)
print(f"RMSE (STEP-E estimator) with a window size of {window_size}: {method_2_rmse_from_first_epochs}")

In [ ]:
y_data = y_data.drop_duplicates(subset=["run_id"])
y_true = y_data["total energy (kJ)"]
y_pred = y_data["estimated total energy (kJ) (GA)"]
# (max gpu power * gpu usage + memory used * W/MB) * training duration
print(f"RMSE (GA): {root_mean_squared_error(y_true, y_pred)}")

In [ ]:
y_pred = y_data["estimated total energy (kJ) (MLCO2)"]
# max gpu power * training duration
print(f"RMSE (MLCO2): {root_mean_squared_error(y_true, y_pred)}")

In [ ]:
if not os.path.exists(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_epoch_start.pkl"):
    groups = metrics.groupby("run_id")
    mean_power_draw_by_epoch_start = {
        "run_id": [],
        "stabilizing epoch": [],
        "window size": [],
        "mean gpu power draw": [],
        "mean ram power draw": [],
        "energy (J)": [],
        "initial energy (J)": [],
        "total epochs": [],
    }
    for run_id, group in tqdm(groups, total=len(groups)):
        n_epochs = group["epoch"].nunique()
        for starting_epoch in np.arange(n_epochs - window_size):
            data = group.query("`epoch` >= @starting_epoch and `epoch` < (@starting_epoch + @window_size)")
            mean_gpu_power_by_epoch_start = data["gpu_power_draw"].mean()
            mean_ram_power_by_epoch_start = data["memory_power_draw"].mean()

            epoch_data = epoch_energy_df.query(
                "`run_id` == @run_id and `epoch` >= @starting_epoch and `epoch` < (@starting_epoch + @window_size)"
            )
            energy = epoch_data["total energy (kJ)"].sum() * KJOULES_TO_JOULES
            initial_energy = (
                epoch_energy_df.query("run_id == @run_id and epoch < @starting_epoch")["total energy (kJ)"].sum()
                * KJOULES_TO_JOULES
            )

            mean_power_draw_by_epoch_start["run_id"].append(run_id)
            mean_power_draw_by_epoch_start["stabilizing epoch"].append(starting_epoch)
            mean_power_draw_by_epoch_start["total epochs"].append(n_epochs)
            mean_power_draw_by_epoch_start["window size"].append(window_size)
            mean_power_draw_by_epoch_start["mean gpu power draw"].append(mean_gpu_power_by_epoch_start)
            mean_power_draw_by_epoch_start["mean ram power draw"].append(mean_ram_power_by_epoch_start)
            mean_power_draw_by_epoch_start["energy (J)"].append(energy)
            mean_power_draw_by_epoch_start["initial energy (J)"].append(initial_energy)

    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_epoch_start.pkl", "wb") as f:
        pickle.dump(mean_power_draw_by_epoch_start, f)
else:
    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_by_epoch_start.pkl", "rb") as f:
        mean_power_draw_by_epoch_start = pickle.load(f)

In [ ]:
energy_estimation_by_epoch_start = pd.DataFrame.from_dict(mean_power_draw_by_epoch_start, orient="columns")
energy_estimation_by_epoch_start = energy_estimation_by_epoch_start.merge(
    aggregated_metrics[
        INDEPENDENT_VARIABLES
        + [
            "run_id",
            "training duration (h)",
            "average gpu power (W)",
            "max power limit (W)",
            "gpu usage (%)",
            "energy (MJ)",
        ]
    ],
    on="run_id",
    how="inner",
)

energy_estimation_by_epoch_start["total energy (kJ)"] = (
    energy_estimation_by_epoch_start["energy (MJ)"] * MJOULES_TO_KJOULES
)
energy_estimation_by_epoch_start["gpu usage (%)"] = energy_estimation_by_epoch_start["gpu usage (%)"] / 100
# energy = window average power * training duration
energy_estimation_by_epoch_start["estimated energy (kJ) (STEP-P)"] = (
    (
        energy_estimation_by_epoch_start["mean gpu power draw"].fillna(0)
        + energy_estimation_by_epoch_start["mean ram power draw"].fillna(0)
    )
    * energy_estimation_by_epoch_start["training duration (h)"]
    * HOURS_TO_SECONDS
    * JOULES_TO_KJOULES
)

# energy = window total energy * #epochs/window size
energy_estimation_by_epoch_start["estimated energy (kJ) (STEP-E)"] = (
    energy_estimation_by_epoch_start["initial energy (J)"]
    + energy_estimation_by_epoch_start["energy (J)"]
    * (energy_estimation_by_epoch_start["total epochs"] - energy_estimation_by_epoch_start["stabilizing epoch"])
    / energy_estimation_by_epoch_start["window size"]
) * JOULES_TO_KJOULES

# energy = (TDP * gpu usage + (ram used * C)) * training duration
energy_estimation_by_epoch_start["estimated energy (kJ) (GA)"] = (
    (
        energy_estimation_by_epoch_start["max power limit (W)"] * energy_estimation_by_epoch_start["gpu usage (%)"]
        + energy_estimation_by_epoch_start["mean ram power draw"].fillna(0)
    )
    * energy_estimation_by_epoch_start["training duration (h)"]
    * HOURS_TO_SECONDS
    * JOULES_TO_KJOULES
)

# energy = TDP * training duration
energy_estimation_by_epoch_start["estimated energy (kJ) (MLCO2)"] = (
    energy_estimation_by_epoch_start["max power limit (W)"]
    * energy_estimation_by_epoch_start["training duration (h)"]
    * HOURS_TO_SECONDS
    * JOULES_TO_KJOULES
)

energy_estimation_by_epoch_start.to_parquet(
    DATA_DIR / "analysis" / "processed" / "energy-estimation-by-epoch-start.gzip",
    compression="gzip",
    index=False,
)

# energy_estimation_by_epoch_start["squared error (STEP-P)"] = (
#     energy_estimation_by_epoch_start["estimated energy (kJ) (STEP-P)"]
#     - energy_estimation_by_epoch_start["energy (kJ)"]
# ) ** 2
# energy_estimation_by_epoch_start["squared error (STEP-E)"] = (
#     energy_estimation_by_epoch_start["estimated energy (kJ) (STEP-E)"]
#     - energy_estimation_by_epoch_start["energy (kJ)"]
# ) ** 2
# energy_estimation_by_epoch_start["squared error (GA)"] = (
#     energy_estimation_by_epoch_start["estimated energy (kJ) (GA)"]
#     - energy_estimation_by_epoch_start["energy (kJ)"]
# ) ** 2
# energy_estimation_by_epoch_start["squared error (MLCO2)"] = (
#     energy_estimation_by_epoch_start["estimated energy (kJ) (MLCO2)"]
#     - energy_estimation_by_epoch_start["energy (kJ)"]
# ) ** 2

In [ ]:
energy_estimation_by_epoch_start.head()

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1)

for i, train_environment in enumerate(energy_estimation_by_epoch_start["training environment"].unique()):
    data = energy_estimation_by_epoch_start.query("`training environment` == @train_environment")
    rmse_method_1 = data.groupby("stabilizing epoch")[["total energy (kJ)", "estimated energy (kJ) (STEP-P)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-P)"])
    )
    rmse_method_2 = data.groupby("stabilizing epoch")[["total energy (kJ)", "estimated energy (kJ) (STEP-E)"]].apply(
        lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-E)"])
    )
    ax0.plot(rmse_method_1.index, rmse_method_1, label=train_environment)
    ax1.plot(rmse_method_2.index, rmse_method_2, label=train_environment)

rmse_method_1 = energy_estimation_by_epoch_start.groupby("stabilizing epoch")[
    ["total energy (kJ)", "estimated energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_by_epoch_start.groupby("stabilizing epoch")[
    ["total energy (kJ)", "estimated energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-E)"]))
ax0.plot(rmse_method_1.index, rmse_method_1, label="All", color="r", linestyle="dashed")
ax1.plot(rmse_method_2.index, rmse_method_2, label="All", color="r", linestyle="dashed")

ax0.legend()
ax0.set_title("STEP-P")
ax0.set_ylabel("RMSE")
ax1.set_title("STEP-E")
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Stabilizing epoch")
stop = energy_estimation_by_epoch_start["stabilizing epoch"].max() + 1
ax0.set_xticks(np.arange(0, stop, 5))
ax1.set_xticks(np.arange(0, stop, 5))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-vs-stabilizing-epoch.{FIGURES_FORMAT}", format=FIGURES_FORMAT)

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1)


rmse_method_1 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_from_first_epochs.groupby("window size")[
    ["total energy (kJ)", "estimated total energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated total energy (kJ) (STEP-E)"]))
ax0.plot(rmse_method_1.index, rmse_method_1, label="Power-based")
ax0.plot(rmse_method_2.index, rmse_method_2, linestyle="--", label="Epoch-energy-based")

ax0.set_title("RMSE of the energy estimation vs. window size")
ax0.legend()
ax0.set_ylabel("RMSE")
ax0.set_xlabel("window size")
stop = energy_estimation_from_first_epochs["window size"].max() + 10
ax0.set_xticks(np.arange(0, stop, 10))
ax0.set_xlim(left=0)

rmse_method_1 = energy_estimation_by_epoch_start.groupby("stabilizing epoch")[
    ["total energy (kJ)", "estimated energy (kJ) (STEP-P)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-P)"]))
rmse_method_2 = energy_estimation_by_epoch_start.groupby("stabilizing epoch")[
    ["total energy (kJ)", "estimated energy (kJ) (STEP-E)"]
].apply(lambda g: root_mean_squared_error(g["total energy (kJ)"], g["estimated energy (kJ) (STEP-E)"]))
ax1.plot(rmse_method_1.index, rmse_method_1, label="STEP-P")
ax1.plot(rmse_method_2.index, rmse_method_2, linestyle="--", label="STEP-E")

ax1.set_title("RMSE of the energy estimation vs. stabilizing epoch")
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Stabilizing epoch")
stop = energy_estimation_by_epoch_start["stabilizing epoch"].max() + 5
ax1.set_xticks(np.arange(0, stop, 10))
ax1.set_xlim(left=0)

if SAVE_FIGS:
    plt.savefig(
        SAVE_FIGS_DIR / f"energy-estimation-vs-window-size-and-stabilizing-epoch.{FIGURES_FORMAT}",
        format=FIGURES_FORMAT,
    )

##### Does it matter which epochs we use to estimate the energy consumption?

Here we will try to estimate the energy consumption of a run based on the power consumption of a random starting epoch.


In [ ]:
if not os.path.exists(DATA_DIR / "analysis" / "processed" / "mean_power_draw_random.pkl"):
    groups = metrics.groupby("run_id")
    mean_power_draw_random = {
        "run_id": [],
        "start epoch": [],
        "window size": [],
        "mean gpu power draw": [],
        "mean ram power draw": [],
        "energy (J)": [],
    }
    for run_id, group in groups:
        stable_epochs = group.query("`epoch` >= @stabilizing_epoch")["epoch"].unique()
        start_epoch = rng.choice(stable_epochs[:-window_size], size=1)[0]
        data = group.query("epoch >= @start_epoch and epoch < (@start_epoch + @window_size)")
        mean_gpu_power = data["gpu_power_draw"].mean()
        mean_ram_power = data["memory_power_draw"].mean()

        epoch_data = epoch_energy_df.query(
            "run_id == @run_id and epoch >= @start_epoch and epoch < (@start_epoch + @window_size)"
        )
        energy = epoch_data["total energy (kJ)"].sum() * KJOULES_TO_JOULES

        mean_power_draw_random["run_id"].append(run_id)
        mean_power_draw_random["start epoch"].append(start_epoch)
        mean_power_draw_random["window size"].append(window_size)
        mean_power_draw_random["mean gpu power draw"].append(mean_gpu_power)
        mean_power_draw_random["mean ram power draw"].append(mean_ram_power)
        mean_power_draw_random["energy (J)"].append(energy)

    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_random.pkl", "wb") as f:
        pickle.dump(mean_power_draw_random, f)
else:
    with open(DATA_DIR / "analysis" / "processed" / "mean_power_draw_random.pkl", "rb") as f:
        mean_power_draw_random = pickle.load(f)

In [ ]:
energy_estimation_from_random = build_energy_estimation(mean_power_draw_random, stabilizing_epoch)
energy_estimation_from_random.head()

In [ ]:
y_data = energy_estimation_from_random
y_true = y_data["total energy (kJ)"]
y_pred = y_data["estimated total energy (kJ) (STEP-P)"]
method_1_rmse_random = root_mean_squared_error(y_true, y_pred)
print(f"RMSE (STEP-P estimator random) with a window size of {window_size}: {method_1_rmse_random}")
y_pred = y_data["estimated total energy (kJ) (STEP-E)"]
method_2_rmse_random = root_mean_squared_error(y_true, y_pred)
print(f"RMSE (STEP-E estimator random) with a window size of {window_size}: {method_2_rmse_random}")

In [ ]:
print(f"Improvement of STEP-P estimator: {method_1_rmse_from_first_epochs - method_1_rmse_random}")
print(f"Improvement of STEP-E estimator: {method_2_rmse_from_first_epochs - method_2_rmse_random}")

As we can see from the previous results, the difference between using the first epoch and a random epoch is negligible.
Hence, to estimate the energy consumption of a run, we only need to measure the power consumption of three epochs at any point after the stabilizing epoch.


In [ ]:
plt.figure(figsize=(20, 5))

x = energy_estimation_from_first_epochs.query("`window size` == 5")["total energy (kJ)"]
y = energy_estimation_from_first_epochs.query("`window size` == 5")["estimated energy (kJ) (STEP-P)"]
_, _, r_value, _, _ = stats.linregress(x, y)
ax = plt.subplot(141)
ax.scatter(x, y)
ax.plot(x, x, color="r", label="y = x")
ax.set_title("STEP-P")
ax.set_xlabel("Real energy (kJ)")
ax.set_ylabel("Estimated energy (kJ)")
ax.text(0.05, 0.9, f"$R^2$: {r_value**2:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
ax.legend()

y = energy_estimation_from_first_epochs.query("`window size` == 5")["estimated energy (kJ) (STEP-E)"]
_, _, r_value, _, _ = stats.linregress(x, y)
ax = plt.subplot(142, sharey=ax)
ax.scatter(x, y)
ax.plot(x, x, color="r", label="y = x")
ax.set_title("STEP-E")
ax.set_xlabel("Real energy (kJ)")
# ax.set_ylabel("Estimated energy (kJ)")
ax.text(0.05, 0.9, f"$R^2$: {r_value**2:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
ax.legend()


y = energy_estimation_from_random.drop_duplicates(subset=["run_id"])["estimated energy (kJ) (GA)"]
_, _, r_value, _, _ = stats.linregress(x, y)
ax = plt.subplot(143, sharey=ax)
ax.scatter(x, y)
ax.plot(x, x, color="r", label="y = x")
ax.set_title("Green Algorithms")
ax.set_xlabel("Real energy (kJ)")
# ax.set_ylabel("Estimated energy (kJ)")
ax.text(0.05, 0.9, f"$R^2$: {r_value**2:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
ax.legend()

y = energy_estimation_from_random.drop_duplicates(subset=["run_id"])["estimated energy (kJ) (MLCO2)"]
_, _, r_value, _, _ = stats.linregress(x, y)
ax = plt.subplot(144, sharey=ax)
ax.scatter(x, y)
ax.plot(x, x, color="r", label="y = x")
ax.set_title("MLCO2 Impact Calculator")
ax.set_xlabel("Real energy (kJ)")
# ax.set_ylabel("Estimated energy (kJ)")
ax.text(0.05, 0.9, f"$R^2$: {r_value**2:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
ax.legend()

plt.tight_layout()

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS_DIR / f"energy-estimation-methods-comparison.{FIGURES_FORMAT}", dpi=300, bbox_inches="tight")